# 56. RTX 3090에서 영상 한 편의 분석이 끝날 때까지

원판과 자세에 같은 합성 영상을 넣고 네 실행 방식의 실제 완료 시간을 비교합니다. 새 Python 프로세스 시작부터 모델 로딩, 116장 디코딩, 두 모델 추론, NMS, 박스 오버레이, H.264 영상과 CSV 저장, 프로세스 종료까지 잽니다. Pod 대여·설치와 TensorRT 엔진 빌드는 일회성 비용으로 따로 기록합니다.

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
from wlcv.config import ROOT

parent = ROOT / "outputs/tensorrt_3090"
complete = sorted(p for p in parent.glob("*") if all((p / f).is_file() for f in ("environment.json", "pose.json", "plate.json", "synthetic_video.json", "precision.json")))
if not complete:
    raise FileNotFoundError("GPU에서 회수한 모델 지연과 합성 영상의 결과 파일을 한 실행 폴더에 두세요.")
folder = complete[-1]
def read(name):
    return json.loads((folder / name).read_text())
environment = read("environment.json")
print(folder.name, environment)
reports = [read("plate.json"), read("pose.json")]

### 영상과 CSV가 저장될 때까지 실제로 몇 초가 걸렸나?

1080×1920 H.264 합성 영상 116장을 이용합니다. 각 방식에 새 프로세스를 두 번 띄웠고 두 번째에는 순서를 뒤집었습니다. 입력 영상은 미리 생성합니다. 결과 영상과 CSV의 장수는 매번 타이머를 멈춘 뒤 검증합니다.

In [ ]:
clips = pd.json_normalize(read("synthetic_video.json"))
if len(clips) != 8 or not (clips.groupby("engine").size() == 2).all():
    raise ValueError("네 방식의 두 번 측정이 모두 있어야 합니다.")
assert (clips[["verified_frames", "verified_csv_rows"]] == 116).all().all()
clips["repeat"] = clips.groupby("engine").cumcount() + 1
seconds = clips.pivot(index="engine", columns="repeat", values="command_s")
seconds.columns = ["첫 실행(s)", "두 번째 실행(s)"]
seconds["평균(s)"] = seconds.mean(axis=1)
seconds["처리량(fps)"] = 116 / seconds["평균(s)"]
display(seconds.round(2))
ax = seconds[["첫 실행(s)", "두 번째 실행(s)"]].set_axis(["First", "Second"], axis=1).plot.barh()
ax.set(xlabel="Seconds from command start to exit", ylabel="")
ax.figure.tight_layout()
ax.figure.savefig(folder / "video_wall_time.png", dpi=155)
plt.show()
stages = clips.groupby("engine")[["command_s", "model_load_s", "seconds", "stages_s.two_models", "stages_s.encoder_pipe"]].mean()
stages["two_model_percent"] = 100 * stages["stages_s.two_models"] / stages["command_s"]
display(stages.round(3))

### 모델 실행과 엔진 빌드 시간

배치 1, 고정 입력 640×384입니다. FP32 기준에서 TF32를 껐습니다. 엔진을 만든 뒤 워밍업하고, 각 모델에서 네 실행 순서를 바꿔 240번씩 측정했습니다. 아래 지연에는 GPU에 있는 입력의 모델 호출과 완료 대기가 들어갑니다. FPS는 측정 횟수를 누적 시간으로 나눈 값입니다.

In [ ]:
rows = []
for report in reports:
    for engine, result in report["timing"].items():
        rows.append({"model": report["model"], "engine": engine, **result})
timing = pd.DataFrame(rows)
display(timing.round(3))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
for (model, group), ax in zip(timing.groupby("model", sort=False), axes):
    group.set_index("engine")[["p50_ms", "p95_ms"]].plot.barh(ax=ax, color=["#216783", "#a4cdd0"])
    ax.set(xlabel="Latency (ms)", ylabel="", title=model)
    ax.invert_yaxis()
fig.savefig(folder / "latency.png", dpi=155)
plt.show()

### 출력 좌표와 점수의 수치 차이

각 출력 후보의 같은 인덱스끼리 PyTorch FP32와의 절대 차이를 구합니다. 좌표는 입력 기준 픽셀이고, 점수는 0~1 범위입니다. 낮은 점수의 후보도 포함합니다. 이 값만으로 실제 원판 검출 mAP나 선수 관절 오차를 계산할 수 없습니다.

In [ ]:
rows = []
for report in reports:
    for engine, groups in report["raw_output_deltas"].items():
        for field, values in groups.items():
            rows.append({"model": report["model"], "engine": engine, "field": field, **values})
display(pd.DataFrame(rows).round(6))

### 같은 TensorRT에서 FP16을 허용하면 출력이 얼마나 달라지나?

같은 12개 텐서를 사용한 FP32와 FP16 후보를 직접 뺍니다. 원판 후보가 0.10, 수행자 후보가 0.25를 넘는지 바뀐 개수도 표시합니다. 난수 입력의 낮은 신뢰도 후보가 대부분이라면 임계값 근처 실제 선수의 판단은 별도로 검증해야 합니다.

In [ ]:
paired = read("precision.json")
pairs = []
for model, comparison in paired.items():
    for field, difference in comparison["raw_deltas"].items():
        pairs.append({"model": model, "field": field, **difference})
display(pd.DataFrame(pairs).round(6))
display(pd.DataFrame([{k: v for k, v in values.items() if k != "raw_deltas"}
                      for values in paired.values()], index=paired).T)

### Pod는 실제로 얼마 동안 열려 있었나?

Runpod 생성 시각부터 종료 후 목록에서 사라진 것을 확인한 시각까지입니다. 대기·설치·빌드·측정·결과 회수가 포함됩니다. 표시한 달러 금액은 시간당 요율로 환산한 GPU 비용 추정입니다.

In [ ]:
if (folder / "lease.json").is_file():
    lease = read("lease.json")
    start = pd.Timestamp(lease["pod_created_utc"]).tz_convert("Asia/Seoul")
    end = pd.Timestamp(lease["termination_verified_utc"]).tz_convert("Asia/Seoul")
    minutes, seconds = divmod(round(lease["measured_seconds"]), 60)
    print(f"{start:%H:%M:%S} → {end:%H:%M:%S} KST | {minutes}분 {seconds}초")
    print(f"기록한 시간당 요율로 계산한 GPU 비용: ${lease['gpu_cost_estimate_usd']:.3f}")
else:
    print("이 실행의 종료 시각이 아직 저장되지 않았습니다.")

### 재현

Mac에서 `wlcv.evaluation.tensorrt_bundle.prepare_bundle`에 새 폴더와 46·47번 참조 폴더를 전달하면 추론용 모델과 측정 코드가 준비됩니다. RTX 3090 Pod에서 같은 버전의 라이브러리를 설치해 `NVIDIA_TF32_OVERRIDE=0 python benchmark.py`를 실행합니다. 이어서 같은 폴더에서 `NVIDIA_TF32_OVERRIDE=0 python synthetic_video.py`를 실행합니다. 모델과 영상은 Git에서 제외하고 결과 JSON만 Mac으로 회수합니다.